# Adaptive Trust Gate — Goodreads *Poetry* (Colab)

Runs the full 7-model pipeline (CF/SVD++, Content-Based, and the Static / Learned /
Bandit / GA / Sequential gates) on the UCSD Book Graph **poetry** slice instead of
MovieLens.

Why poetry and not young_adult: the genre dumps all share one schema, so the only
difference is scale. Poetry is ~2.73M interactions across 36,514 books (145MB gzipped);
young_adult is ~34.6M interactions (1.7GB), which is far past what Surprise's SVD++ can
train in a Colab session. Poetry keeps the Goodreads framing and the natural sparsity
while staying tractable.

Files download straight from UCSD — no Drive mount needed.

**Runtime → Change runtime type → High-RAM** is recommended. A GPU only helps Model 7
(the BiLSTM gate); everything else is CPU-bound.


## 1. Clone the repo and install dependencies


In [ ]:
!git clone -q https://github.com/Ar555Rathod/adaptive-trust-gate.git
%cd adaptive-trust-gate


In [ ]:
# scikit-surprise builds Cython extensions against the installed NumPy and is the one
# dependency that regularly fails on current Colab images. Install and verify it up front
# rather than discovering the problem after downloading the dataset.
!pip install -q -r requirements.txt

try:
    import surprise
    print('scikit-surprise OK:', surprise.__version__)
except Exception as e:
    print('scikit-surprise FAILED to import:', e)
    print('Fix: run the fallback cell below, then Runtime -> Restart session.')


In [ ]:
# FALLBACK -- only run this if the import above failed, then restart the runtime.
# Surprise 1.1.x needs a NumPy 1.x ABI to build its Cython extensions.
# !pip install -q "numpy<2" && pip install -q --no-binary :all: scikit-surprise


## 2. Download the Goodreads poetry dataset

Direct from the McAuley Lab host. ~172MB total, usually well under a minute on Colab.

Note: the older `datarepo.eng.ucsd.edu` URLs that circulate in blog posts now 404 —
`mcauleylab.ucsd.edu` is the live mirror.


In [ ]:
BASE = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre'

!mkdir -p data/raw/goodreads
!wget -q --show-progress -O data/raw/goodreads/goodreads_books_poetry.json.gz \
    {BASE}/goodreads_books_poetry.json.gz
!wget -q --show-progress -O data/raw/goodreads/goodreads_interactions_poetry.json.gz \
    {BASE}/goodreads_interactions_poetry.json.gz

!ls -lh data/raw/goodreads


In [ ]:
# Sanity check: both files should gunzip and the first line should be valid JSON.
import gzip, json

for name in ('books', 'interactions'):
    path = f'data/raw/goodreads/goodreads_{name}_poetry.json.gz'
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        rec = json.loads(f.readline())
    print(name, '->', sorted(rec)[:8], '...')


## 3. Point the pipeline at Goodreads poetry

`ATG_DATASET` selects the normalizer; `ATG_GOODREADS_GENRE` selects which genre slice it
reads. Outputs are keyed on both, so results land in `results/goodreads_poetry/` and can't
collide with a MovieLens or young_adult run.


In [ ]:
%env ATG_DATASET=goodreads
%env ATG_GOODREADS_GENRE=poetry
%env PYTHONPATH=src

!python -c "from atg import config; print('raw      :', config.RAW_DIR); \
print('processed:', config.PROCESSED_DIR); print('results  :', config.RESULTS_DIR)"


## 4. Normalize and build the train/val/test split

Drops unrated shelf entries (`want to read` / `currently reading` carry `rating: 0`),
parses `date_added` into Unix timestamps, and builds book metadata text from title +
description for the content-based expert.


In [ ]:
!python src/atg/data/normalize.py


In [ ]:
!python scripts/01_build_splits.py


### Check the sparsity segments before training

This is the cell worth pausing on. The whole Adaptive Trust Gate hypothesis is that the
gate helps most for **cold** users, so if the cold segment comes out tiny or empty, the
rest of the run can't test the hypothesis and you should stop here.

Unlike MovieLens (which guarantees >=20 ratings per user), Goodreads has genuinely sparse
users — so a large cold segment here is *real*, not simulated.


In [ ]:
# Paths come from atg.config so this cell is correct whether or not the repo
# has the ATG_GOODREADS_GENRE patch (outputs land in goodreads_poetry/ if it does,
# goodreads/ if it doesn't).
import sys
sys.path.insert(0, 'src')
import pandas as pd
from atg import config

PROC = config.PROCESSED_DIR
METRICS = config.METRICS_DIR
print('reading from:', PROC, '\n')

segs = pd.read_csv(PROC / 'user_segments.csv')
print(segs['segment'].value_counts(), '\n')
print('users:', len(segs))

for split in ('train', 'val', 'test'):
    df = pd.read_csv(PROC / f'{split}.csv')
    print(f'{split:>5}: {len(df):>9,} ratings')


## 5. Train the two experts

**This is the slowest cell in the notebook** — SVD++ is single-threaded Cython and its
cost grows with the square of each user's rating count. Expect tens of minutes. Keep the
tab open; Colab disconnects idle sessions.


In [ ]:
!python scripts/02_train_experts.py


## 6. Run all five gates, then the full comparison


In [ ]:
!python scripts/03_static_hybrid.py
!python scripts/04_learned_gate.py


In [ ]:
!python scripts/05_bandit_gate.py
!python scripts/06_ga_gate.py


In [ ]:
!python scripts/07_sequential_gate.py


### External baselines

Models 1-7 are all ours, so on their own they show which gate is best but not
whether gating beats the published alternatives. This adds four standard combiners
-- switching hybrid (Burke 2002), feature-weighted linear stacking (Sill et al. 2009),
ridge stacking, and gradient-boosted stacking -- fit on the same VAL partition the
gates use. They reuse the cached expert predictions, so this runs in minutes.

Must run **after** `02_train_experts.py` and **before** `08_full_comparison.py`.


In [ ]:
!python scripts/11_external_baselines.py


In [ ]:
!python scripts/08_full_comparison.py


## 7. Results


In [ ]:
import pandas as pd

print('reading from:', METRICS, '\n')

print('--- FULL COMPARISON (by sparsity segment) ---')
display(pd.read_csv(METRICS / 'full_comparison_table.csv'))

print('--- STATISTICAL SIGNIFICANCE (Model 3 vs adaptive gates) ---')
display(pd.read_csv(METRICS / 'full_comparison_significance.csv'))


## 8. Interpretability plots


In [ ]:
!python scripts/09_interpretability.py


In [ ]:
from IPython.display import Image, display

for caption, fname in [
    ('Interpretability vs accuracy (Model 4 vs Model 7)', 'interpretability_vs_accuracy.png'),
    ('Bandit gate learning curve',                        'model5_learning_curve.png'),
    ('GA fitness curve',                                  'model6_ga_fitness_curve.png'),
    ('Learned gate: trust vs user sparsity',              'model4_g_vs_sparsity.png'),
]:
    print(caption)
    display(Image(str(METRICS / fname)))


## 9. Save results before the runtime dies

`results/` is gitignored and Colab wipes local files on disconnect, so copy the metrics
out. The whole metrics directory is small (CSV/JSON/PNG) — the trained `.pkl` models are
the bulky part and are skipped here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/atg_results
!cp -r {config.RESULTS_DIR} /content/drive/MyDrive/atg_results/
!ls -R /content/drive/MyDrive/atg_results | head -40


## Optional: multi-seed robustness

`scripts/10_multiseed_full.py` re-runs the *entire* pipeline across 5 seeds — including
5 full SVD++ trainings. Only start this once the single-seed run above has completed and
you know how long section 5 took; multiply that by roughly five.


In [ ]:
# !python scripts/10_multiseed_full.py
